# Entregable Hito 0: Definición del Problema y Estrategia de Variables
## PFC - Sistema Inteligente de Precios Dinámicos
**Autor:** Ricardo Fernández 
---
**Nota**: Este no es el cuaderno final se trata de una pequeña guia en la que se ve y contesta a las cuestiones que se requiere para este entonces. Al final o a medida que se acerque compleción o la concatenación de notebooks con los diferentes hitos se creará una versión final que tendrá


---

## 1. Contexto del Problema y Utilidad Real

### 1.1 El Escenario Macroeconómico del Sector Alquiler Vacacional

El mercado de los alojamientos turísticos y de corto plazo ha experimentado una disrupción tecnológica sin precedentes, consolidándose como uno de los pilares más dinámicos del ecosistema **PropTech**. Sin embargo, esta maduración del mercado ha traído consigo un entorno operativo de **extrema volatilidad, marcado por una estacionalidad agresiva y una competencia hiperfragmentada**.

En la actualidad, la fijación de precios en las plataformas de alquiler vacacional adolece de un problema endémico: **la rigidez o la heurística intuitiva**. Tradicionalmente, la gran mayoría de los propietarios o pequeños gestores de activos determinan sus tarifas basándose en tres metodologías subóptimas:

* **Intuición empírica ("gut feeling"):** Fijación de precios basada en percepciones sesgadas del valor de su propia propiedad.
* **Medias históricas estáticas:** Replicar las tarifas del año anterior aplicando un porcentaje plano de inflación, ignorando los microcambios macroeconómicos del presente.
* **Estrategias espejo pasivas:** Copiar a ciegas los precios de los competidores más cercanos, asumiendo que sus costes de estructura y sus tasas de ocupación son idénticos.

Esta desconexión con la realidad analítica del mercado genera **ineficiencias operacionales críticas**:

* **El coste de la sobrevaloración (Overpricing):** Establecer un precio por encima del umbral de tolerancia del mercado resulta en ventanas de vaciado prolongadas. Un inmueble vacío no solo deja de facturar ingresos directos, sino que sigue devengando costes fijos (mantenimiento, suministros, amortizaciones), destruyendo el margen de beneficio neto de la explotación.
* **El coste de oportunidad por infravaloración (Underpricing):** Establecer precios por debajo de la disposición máxima a pagar del cliente (*willingness to pay*), especialmente en momentos de picos de demanda (festivos, macroeventos locales, ferias internacionales), se traduce en "dejar dinero sobre la mesa". El alojamiento se reserva con demasiada antelación, perdiendo la capacidad de capturar el valor real de esa ventana temporal.

---

### 1.2 La Utilidad Real del Proyecto: Optimización Avanzada del RevPAR

La utilidad real de este proyecto de fin de curso no radica únicamente en construir un estimador matemático, sino en diseñar e implementar un **Sistema de Precios Dinámicos Inteligente** que actúe como un motor de *Revenue Management* automatizado. La métrica reina que este sistema busca maximizar es el **RevPAR (Revenue per Available Room)**, el indicador de rendimiento estándar en la industria hotelera y turística, cuya fórmula conceptual se define como:

$$\text{RevPAR} = \text{Tasa de Ocupación} \times \text{Precio Medio Diario (ADR)}$$

Fijar el precio óptimo exige romper con el paradigma de las bases de datos aisladas. Para maximizar el RevPAR, este sistema unifica por primera vez **tres dimensiones críticas del mercado que operan a velocidades y estructuras radicalmente distintas**, consolidando una verdadera visión de 360 grados del ecosistema del inmueble:

* **1. La Infraestructura Física (Dimensión Estática/Estructurada):** Representa las características intrínsecas e inmutables del activo inmobiliario (tipología de propiedad, capacidad de huéspedes, número de habitaciones, camas y baños). Esta información, extraída de sistemas relacionales tradicionales (**Amazon RDS**), define el *precio suelo o coste marginal de explotación*, el umbral mínimo por debajo del cual el inmueble no es rentable.
* **2. La Reputación Digital y Calidad Percibida (Dimensión Semiestructurada/Textual):** Las métricas de valoración tradicionales (las clásicas puntuaciones de 1 a 5 estrellas) son métricas burdas que sufren de inflación de calificaciones. La verdadera ventaja competitiva y el valor diferencial percibido por el cliente se esconden en el texto libre de las reseñas. Mediante técnicas de **Procesamiento de Lenguaje Natural (NLP)** sobre almacenes documentales (**MongoDB Atlas**), el sistema extrae un *score de sentimiento* que cuantifica factores intangibles como la hospitalidad, el ruido, la limpieza real o la comodidad del check-in, permitiendo aplicar un "premium" de precio justificado por la experiencia del usuario.
* **3. El Pulso del Mercado en Tiempo Real (Dimensión Dinámica/Streaming):** La demanda en el sector PropTech es extremadamente líquida; varía minuto a minuto según el volumen de usuarios concurrentes buscando alojamiento en una zona geográfica concreta. Capturar este comportamiento antes de que se formalice la reserva es vital. Utilizando una arquitectura de eventos distribuidos (**Apache Kafka**), el sistema mide la telemetría web (clics, búsquedas, visualizaciones), abstrayendo un índice de presión de la demanda que fuerza al modelo predictivo a ajustar las tarifas de manera elástica ante picos repentinos de interés.

---

### 1.3 Impacto en el Negocio y Automatización

El impacto directo de la implementación de esta arquitectura híbrida es la **democratización y automatización de las decisiones de pricing**. Al delegar la analítica de datos en un pipeline unificado que consolida la información en un Data Lake inmutable (**Amazon S3**), eliminamos el sesgo humano del proceso.

El sistema dota a la plataforma de una agilidad algorítmica capaz de autoajustar las tarifas de cientos de inmuebles de forma simultánea y en tiempo real. Esto no solo incrementa drásticamente la rentabilidad neta del propietario, sino que posiciona a la plataforma tecnológica en un nivel superior de competitividad dentro del sector turístico digital.

## 2. Definición del Problema de Machine Learning

Desde la perspectiva analítica, el desafío se clasifica formalmente como un problema de **Aprendizaje Supervisado** en la tarea de **Regresión**.

* **Por qué es Supervisado:** Disponemos de un conjunto de datos históricos etiquetados donde cada patrón de entrada se asocia a un coste real verificado en el mercado.
* **Por qué es Regresión:** La variable que buscamos predecir es numérica, continua y flotante (un valor monetario en Euros). El objetivo matemático del algoritmo será entrenar una función estimadora que minimice métricas de error como el Error Absoluto Medio (MAE) o el Error Cuadrático Medio (MSE).

## 3. Identificación y Justificación de las Variables

Para garantizar la precisión del modelo, se ha diseñado una matriz de variables enriquecida que proviene de tres fuentes de datos independientes estructuradas bajo la llave común `listing_id`:

### 3.1 Variable Objetivo (Target)
* **`precio_noche` (Numérica Continua):** Representa la tarifa óptima recomendada por noche. Es la métrica económica que el negocio necesita ajustar dinámicamente.

### 3.2 Variables Independientes (Features)

| Variable | Tipo de Dato | Origen / Fuente | Justificación de Negocio |
| :--- | :--- | :--- | :--- |
| **`property_type`** | Categórica | Amazon RDS (Estructurado) | Determina el segmento de mercado base y el perfil de cliente (Estudio, Villa, Apartamento). |
| **`accommodates`** | Numérica Entera | Amazon RDS (Estructurado) | Capacidad total de huéspedes. Es el correlacionador físico de mayor peso con el coste base. |
| **`beds` / `bedrooms`** | Numérica Entera | Amazon RDS (Estructurado) | Define la distribución del inmueble, clave para la logística de familias o grupos. |
| **`comments`** | Texto Libre | MongoDB Atlas (NoSQL) | Contenido de las opiniones de los usuarios. Clave para extraer detalles cualitativos mediante NLP. |
| **`score_sentimiento`** | Numérica Continua | MongoDB -> NLP Extracción | Variable derivada ([0.0 a 1.0]). Permite premiar o castigar el precio según la reputación online. |
| **`ratio_busquedas_zona`** | Numérica Continua | Apache Kafka (Streaming) | Índice dinámico de demanda basado en los clics concurrentes de usuarios en la web. |

## 4. Integración Multifuente e Información Híbrida

La innovación técnica de este pipeline radica en que integra datos con tres estructuras y velocidades de ingesta completamente diferentes:
1. **Estructurada (SQL/RDS):** Almacena el inventario maestro de las propiedades.
2. **Semiestructurada (NoSQL/MongoDB Atlas):** Alberga las reseñas masivas en documentos JSON flexibles.
3. **Streaming en Tiempo Real (Apache Kafka):** Captura el flujo constante de búsquedas antes de que se consoliden en reservas.

A continuación, se presenta la celda de código que ejecuta la simulación de integración híbrida utilizando **Pandas** para consolidar las tres fuentes en una única 'Súper Tabla' analítica que será persistida en el Data Lake de **Amazon S3**.

In [ ]:
import pandas as pd
import json

print("--- INICIANDO PROCESO DE INTEGRACIÓN MULTIFUENTE HITO 0/1 ---")

# 1. Simulación de datos físicos desde la base de datos relacional (RDS)
rds_listings = [
    {"listing_id": 101, "property_type": "Apartamento", "accommodates": 4, "precio_noche": 120.0},
    {"listing_id": 102, "property_type": "Estudio", "accommodates": 2, "precio_noche": 85.0}
]
df_rds = pd.DataFrame(rds_listings)

# 2. Simulación de opiniones cualitativas desde MongoDB Atlas
mongo_reviews = [
    {"listing_id": 101, "comments": "Excelente ubicación y muy limpio.", "score_sentimiento": 0.92},
    {"listing_id": 102, "comments": "Un poco ruidoso por las noches.", "score_sentimiento": 0.51}
]
df_mongo = pd.DataFrame(mongo_reviews)

# 3. Simulación de clics de demanda en tiempo real desde Apache Kafka
kafka_stream = [
    {"listing_id": 101, "ratio_busquedas_zona": 1.4},
    {"listing_id": 101, "ratio_busquedas_zona": 1.6},
    {"listing_id": 102, "ratio_busquedas_zona": 0.9}
]
df_kafka = pd.DataFrame(kafka_stream)

# Agregación estadística de las fuentes dinámicas antes del acoplamiento
df_kafka_agg = df_kafka.groupby('listing_id')['ratio_busquedas_zona'].mean().reset_index()

# JOIN Lógico utilizando 'listing_id' como llave de unificación
df_master = df_rds.merge(df_mongo[['listing_id', 'score_sentimiento']], on='listing_id', how='left')
df_master = df_master.merge(df_kafka_agg, on='listing_id', how='left')

print("\n✔ ¡Dataset unificado generado con éxito para la fase de Machine Learning!")
print("Estructura final del Data Lake (Muestra):")
df_master.head()

: 